In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

In [ ]:
path = kagglehub.dataset_download("surajghuwalewala/ham1000-segmentation-and-classification")
print("Path to dataset files:", path)

In [ ]:
def create_linear_kernel(length, angle_degrees):
    """
    Creates a linear structuring element of a specified length and angle for morphological operations.

    Parameters:
    - length: odd integer (e.g., 9, 11, 15). Defines the size of the matrix.
    - angle_degrees: float or int (0-180). Angle of the line in degrees.
    """
    kernel = np.zeros((length, length), dtype=np.uint8)
    center = length // 2
    angle_rad = np.radians(angle_degrees)
    x_offset = int(np.round((length / 2) * np.cos(angle_rad)))
    y_offset = int(np.round((length / 2) * np.sin(angle_rad)))
    p1 = (center - x_offset, center - y_offset)
    p2 = (center + x_offset, center + y_offset)
    cv2.line(kernel, p1, p2, 1, 1)
    
    return kernel

In [ ]:
def filter_mask_with_opening(rough_mask, kernel_length=11):
    """Uses directional opening to clean the rough mask of hair pixels, preserving circular features like globules."""
    clean_mask = np.zeros_like(rough_mask)
    angles = [0, 30, 60, 90, 120, 150]
    for angle in angles:
        kernel = create_linear_kernel(kernel_length, angle)
        opened_mask = cv2.morphologyEx(rough_mask, cv2.MORPH_OPEN, kernel)
        clean_mask = cv2.max(clean_mask, opened_mask)
    
    return clean_mask

In [ ]:
def hair_removal(img_rgb):
    """
    Remove hair from the input RGB image using morphological operations and inpainting.
    """
    cross_kernel_length = 15
    opening_kernel_length = 15
    threshold_value = 15
    max_value = 255
    dilate_kernel_length = 3
    inpaint_radius = 3

    r_channel = img_rgb[:, :, 0]

    kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (cross_kernel_length, cross_kernel_length))
    blackhat = cv2.morphologyEx(r_channel, cv2.MORPH_BLACKHAT, kernel_cross)

    _, rough_mask = cv2.threshold(blackhat, threshold_value, max_value, cv2.THRESH_BINARY)
    clean_mask = filter_mask_with_opening(rough_mask, kernel_length=opening_kernel_length)

    kernel_dilate = np.ones((dilate_kernel_length, dilate_kernel_length), np.uint8)
    mask_dilated = cv2.dilate(clean_mask, kernel_dilate, iterations=1)
    
    inpainted = cv2.inpaint(img_rgb, mask_dilated, inpaintRadius=inpaint_radius, flags=cv2.INPAINT_NS)

    return mask_dilated, inpainted

In [ ]:
def extract_dimension_features(contour, image_shape):
    """
    Computes dimension-related features from the original contour of the lesion.
    """
    lesion_area = cv2.contourArea(contour)
    total_image_area = image_shape[0] * image_shape[1]
    relative_area = lesion_area / total_image_area if total_image_area > 0 else 0.0
    eq_diameter = np.sqrt((4 * lesion_area) / np.pi) if lesion_area > 0 else 0.0
    if len(contour) >= 5:
        (_, _), (_, major_axis), _ = cv2.fitEllipse(contour)
    else:
        major_axis = 0.0
        
    return relative_area, eq_diameter, major_axis

In [ ]:
def preprocess_crop_and_resize(image_rgb, binary_mask, target_size=(256, 256)):
    """
    Preprocesses the input image and mask by performing:
    1. Cropping the image and mask to the bounding box of the lesion, with an optional padding to avoid cutting too close to the edges.
    2. Resizing the cropped image and mask to the target size, using appropriate interpolation methods to maintain image quality.   
    """
    padding_factor = 0.10

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        padding_x = int(w * padding_factor)
        padding_y = int(h * padding_factor)
        x_start = max(0, x - padding_x)
        y_start = max(0, y - padding_y)
        x_end = min(image_rgb.shape[1], x + w + padding_x)
        y_end = min(image_rgb.shape[0], y + h + padding_y)

        cropped_image = image_rgb[y_start:y_end, x_start:x_end]
        cropped_mask = binary_mask[y_start:y_end, x_start:x_end]
    else:
        cropped_image = image_rgb
        cropped_mask = binary_mask

    if cropped_image.size == 0 or cropped_mask.size == 0:
        return cv2.resize(image_rgb, target_size), cv2.resize(binary_mask, target_size)

    crop_h, crop_w = cropped_image.shape[:2]
    max_side = max(crop_h, crop_w)
    top = (max_side - crop_h) // 2
    bottom = max_side - crop_h - top
    left = (max_side - crop_w) // 2
    right = max_side - crop_w - left

    square_image = cv2.copyMakeBorder(cropped_image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0,0,0])
    square_mask = cv2.copyMakeBorder(cropped_mask, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

    resized_image = cv2.resize(square_image, target_size, interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(square_mask, target_size, interpolation=cv2.INTER_NEAREST)

    return resized_image, resized_mask

In [ ]:
def compute_fourier_descriptors(contour, num_descriptors=15):
    """
    Computes the Fourier Descriptors for a given contour based on the distance from the centroid.
    """
    M = cv2.moments(contour)
    if M['m00'] == 0:
        return np.zeros(num_descriptors)
    cx = M['m10'] / M['m00']
    cy = M['m01'] / M['m00']
    x = contour[:, 0, 0].astype(float)
    y = contour[:, 0, 1].astype(float)
    distances = np.sqrt((x - cx)**2 + (y - cy)**2)
    
    fft_result = np.fft.fft(distances)
    magnitudes = np.abs(fft_result)
    if len(magnitudes) <= 1:
        return np.zeros(num_descriptors)
    fundamental_magnitude = magnitudes[1]
    if fundamental_magnitude == 0:
        fundamental_magnitude = 1e-5
    fd_normalized = magnitudes[1 : num_descriptors + 1] / fundamental_magnitude
    if len(fd_normalized) < num_descriptors:
        pad = np.zeros(num_descriptors - len(fd_normalized))
        fd_normalized = np.concatenate([fd_normalized, pad])
        
    return fd_normalized

In [ ]:
def calculate_asymmetry_index(binary_mask):
    """
    Compute the Geometric Asymmetry Index of a binary mask by:
    1. Finding the largest contour to focus on the main lesion.
    2. Rotating the lesion to align with its principal axes using minAreaRect.
    3. Cropping the aligned lesion to its bounding box.
    4. Flipping the cropped lesion horizontally and vertically, then calculating the XOR to find non-overlapping pixels.
    5. Returning asym_x and asym_y indices, where values close to 0 indicate symmetry and values close to 1 indicate high asymmetry.
    """
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0, 0.0 
        
    c = max(contours, key=cv2.contourArea)
    area_totale = cv2.contourArea(c)
    
    if area_totale == 0:
        return 0.0, 0.0

    rect = cv2.minAreaRect(c)
    (center_x, center_y), (width, height), angle = rect
    
    # align to horizontal/vertical axes
    if width < height:
        angle += 90
        
    M = cv2.getRotationMatrix2D((center_x, center_y), angle, 1.0)
    aligned_mask = cv2.warpAffine(binary_mask, M, (binary_mask.shape[1], binary_mask.shape[0]))
    
    x, y, w, h = cv2.boundingRect(cv2.findNonZero(aligned_mask))
    cropped_aligned = aligned_mask[y:y+h, x:x+w]
    
    if cropped_aligned.size == 0 or w == 0 or h == 0:
        return 0.0, 0.0

    # compute asymmetry indices 
    flipped_h = cv2.flip(cropped_aligned, 1)
    xor_h = cv2.bitwise_xor(cropped_aligned, flipped_h)
    asym_x = np.count_nonzero(xor_h) / (2 * area_totale)
    
    flipped_v = cv2.flip(cropped_aligned, 0)
    xor_v = cv2.bitwise_xor(cropped_aligned, flipped_v)
    asym_y = np.count_nonzero(xor_v) / (2 * area_totale)
    
    return asym_x, asym_y

In [ ]:
def compute_glcm_energy(image_gray, segmentation_mask, levels=256):
    """
    Calculates the GLCM Energy feature for a given grayscale image and its corresponding binary segmentation mask.
    """
    directions = [
        (1, 0),
        (1, -1),
        (0, -1),
        (-1, -1)
    ]
    energies = []
    for dx, dy in directions:
        if dy > 0:
            v1_y, v2_y = slice(0, -dy), slice(dy, None)
        elif dy < 0:
            v1_y, v2_y = slice(-dy, None), slice(0, dy)
        else:
            v1_y, v2_y = slice(None), slice(None)
        if dx > 0:
            v1_x, v2_x = slice(0, -dx), slice(dx, None)
        elif dx < 0:
            v1_x, v2_x = slice(-dx, None), slice(0, dx)
        else:
            v1_x, v2_x = slice(None), slice(None)
            
        img_v1 = image_gray[v1_y, v1_x]
        img_v2 = image_gray[v2_y, v2_x]
        mask_v1 = segmentation_mask[v1_y, v1_x] > 0
        mask_v2 = segmentation_mask[v2_y, v2_x] > 0
        
        valid_pairs = mask_v1 & mask_v2
        p1 = img_v1[valid_pairs].astype(np.int32)
        p2 = img_v2[valid_pairs].astype(np.int32)
        if len(p1) == 0:
            energies.append(0.0)
            continue
        
        linear_indices = p1 * levels + p2
        glcm_1d = np.bincount(linear_indices, minlength=levels*levels)
        glcm = glcm_1d.reshape((levels, levels)).astype(np.float64)
        
        total_pairs = np.sum(glcm)
        if total_pairs > 0:
            glcm_norm = glcm / total_pairs
            energy = np.sum(glcm_norm ** 2)
            energies.append(energy)
        else:
            energies.append(0.0)
    
    return np.mean(energies) if energies else 0.0

In [ ]:
from skimage.feature import local_binary_pattern

def extract_lbp_features(image_gray, segmentation_mask, radius=1, n_points=8):
    """
    Calculates the uniform LBP histogram features for the pixels within the lesion area defined by the segmentation mask.
    """
    lbp = local_binary_pattern(image_gray, n_points, radius, method='uniform')
    lbp_lesion = lbp[segmentation_mask > 0]
    if lbp_lesion.size == 0:
        return np.zeros(n_points + 2)
    
    n_bins = int(n_points + 2)
    hist, _ = np.histogram(lbp_lesion, bins=n_bins, range=(0, n_bins), density=True)
    
    return hist